# 04 - Validaciones y exploracion

Objetivo: documentar calidad minima, coherencia temporal, cobertura por lote y auditoria de ingesta para el PSet 3.

In [13]:
import os
from pyspark.sql import SparkSession


In [14]:
spark_snowflake_pkg = os.getenv('SPARK_SNOWFLAKE_PACKAGE', 'net.snowflake:spark-snowflake_2.12:3.1.8')
snowflake_jdbc_pkg = os.getenv('SNOWFLAKE_JDBC_PACKAGE', 'net.snowflake:snowflake-jdbc:3.14.4')

spark = (
    SparkSession.builder
    .appName('PSet3 Validaciones y Exploracion')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.jars.packages', f"{spark_snowflake_pkg},{snowflake_jdbc_pkg}")
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)

print(spark.version)

3.5.0


## 1) Configuracion

In [15]:
def get_env(name, default=None, required=False):
    value = os.getenv(name, default)
    if required and (value is None or str(value).strip() == ''):
        raise ValueError(f'Falta variable de entorno requerida: {name}')
    return value

snowflake_host = get_env('SNOWFLAKE_HOST', '').strip()
snowflake_account = get_env('SNOWFLAKE_ACCOUNT', '').strip()
snowflake_port = get_env('SNOWFLAKE_PORT', '443').strip()

if not snowflake_host:
    if not snowflake_account:
        raise ValueError('Debes configurar SNOWFLAKE_HOST o SNOWFLAKE_ACCOUNT')
    snowflake_host = f"{snowflake_account}.snowflakecomputing.com"

if snowflake_port and snowflake_port != '443':
    snowflake_host = f"{snowflake_host}:{snowflake_port}"

sf_options = {
    'sfURL': snowflake_host,
    'sfUser': get_env('SNOWFLAKE_USER', required=True),
    'sfPassword': get_env('SNOWFLAKE_PASSWORD', required=True),
    'sfRole': get_env('SNOWFLAKE_ROLE', required=True),
    'sfWarehouse': get_env('SNOWFLAKE_WAREHOUSE', required=True),
    'sfDatabase': get_env('SNOWFLAKE_DATABASE', required=True),
    'sfSchema': get_env('SNOWFLAKE_SCHEMA_ANALYTICS', 'ANALYTICS'),
}

raw_schema = get_env('SNOWFLAKE_SCHEMA_RAW', 'RAW')
analytics_schema = get_env('SNOWFLAKE_SCHEMA_ANALYTICS', 'ANALYTICS')
run_id = get_env('RUN_ID', 'manual_local_001')

obt_table = f"{analytics_schema}.OBT_TRIPS"
audit_table = f"{raw_schema}.INGESTION_AUDIT"

{k: ('***' if 'Password' in k else v) for k, v in sf_options.items()}

{'sfURL': 'DVNRQOK-CRC15844.snowflakecomputing.com',
 'sfUser': 'tonyfajardo1d',
 'sfPassword': '***',
 'sfRole': 'ACCOUNTADMIN',
 'sfWarehouse': 'PSET3_WH',
 'sfDatabase': 'DM_PSET3',
 'sfSchema': 'ANALYTICS'}

## 2) Reglas de calidad minima (nulos esenciales)

In [16]:
q_nulls = f"""
SELECT
  SUM(CASE WHEN trip_nk IS NULL THEN 1 ELSE 0 END) AS null_trip_nk,
  SUM(CASE WHEN pickup_datetime IS NULL THEN 1 ELSE 0 END) AS null_pickup_datetime,
  SUM(CASE WHEN dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS null_dropoff_datetime,
  SUM(CASE WHEN pu_location_id IS NULL THEN 1 ELSE 0 END) AS null_pu_location_id,
  SUM(CASE WHEN do_location_id IS NULL THEN 1 ELSE 0 END) AS null_do_location_id,
  SUM(CASE WHEN service_type IS NULL THEN 1 ELSE 0 END) AS null_service_type
FROM {obt_table}
"""

(
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_nulls)
    .load()
    .show(truncate=False)
)

+------------+--------------------+---------------------+-------------------+-------------------+-----------------+
|NULL_TRIP_NK|NULL_PICKUP_DATETIME|NULL_DROPOFF_DATETIME|NULL_PU_LOCATION_ID|NULL_DO_LOCATION_ID|NULL_SERVICE_TYPE|
+------------+--------------------+---------------------+-------------------+-------------------+-----------------+
|0           |0                   |0                    |0                  |0                  |0                |
+------------+--------------------+---------------------+-------------------+-------------------+-----------------+



## 3) Reglas de rangos logicos

In [17]:
q_ranges = f"""
SELECT
  SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END) AS negative_trip_distance,
  SUM(CASE WHEN trip_distance > 100 THEN 1 ELSE 0 END) AS too_high_trip_distance,
  SUM(CASE WHEN trip_duration_min < 0 THEN 1 ELSE 0 END) AS negative_trip_duration,
  SUM(CASE WHEN trip_duration_min > 240 THEN 1 ELSE 0 END) AS too_high_trip_duration,
  SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END) AS negative_total_amount,
  SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) AS negative_fare_amount,
  SUM(CASE WHEN avg_speed_mph < 0 THEN 1 ELSE 0 END) AS negative_avg_speed,
  SUM(CASE WHEN avg_speed_mph > 120 THEN 1 ELSE 0 END) AS too_high_avg_speed,
  SUM(CASE WHEN passenger_count IS NOT NULL AND (passenger_count < 1 OR passenger_count > 6) THEN 1 ELSE 0 END) AS invalid_passenger_count
FROM {obt_table}
"""

(
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_ranges)
    .load()
    .show(truncate=False)
)

+----------------------+----------------------+----------------------+----------------------+---------------------+--------------------+------------------+------------------+-----------------------+
|NEGATIVE_TRIP_DISTANCE|TOO_HIGH_TRIP_DISTANCE|NEGATIVE_TRIP_DURATION|TOO_HIGH_TRIP_DURATION|NEGATIVE_TOTAL_AMOUNT|NEGATIVE_FARE_AMOUNT|NEGATIVE_AVG_SPEED|TOO_HIGH_AVG_SPEED|INVALID_PASSENGER_COUNT|
+----------------------+----------------------+----------------------+----------------------+---------------------+--------------------+------------------+------------------+-----------------------+
|0                     |0                     |0                     |0                     |0                    |0                   |0                 |0                 |0                      |
+----------------------+----------------------+----------------------+----------------------+---------------------+--------------------+------------------+------------------+-----------------------+



## 4) Coherencia temporal PU/DO

In [18]:
q_temporal = f"""
SELECT
  SUM(CASE WHEN dropoff_datetime < pickup_datetime THEN 1 ELSE 0 END) AS invalid_temporal_order,
  MIN(trip_duration_min) AS min_trip_duration_min,
  MAX(trip_duration_min) AS max_trip_duration_min,
  SUM(CASE WHEN ABS(year - TRY_TO_NUMBER(source_year)) > 1 THEN 1 ELSE 0 END) AS inconsistent_year_vs_source
FROM {obt_table}
"""

(
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_temporal)
    .load()
    .show(truncate=False)
)

+----------------------+---------------------+---------------------+---------------------------+
|INVALID_TEMPORAL_ORDER|MIN_TRIP_DURATION_MIN|MAX_TRIP_DURATION_MIN|INCONSISTENT_YEAR_VS_SOURCE|
+----------------------+---------------------+---------------------+---------------------------+
|0                     |1.0                  |240.0                |0                          |
+----------------------+---------------------+---------------------+---------------------------+



## 5) Cobertura por servicio/anio/mes (OBT)

In [19]:
q_coverage = f"""
SELECT service_type, source_year, source_month, COUNT(*) AS rows_obt
FROM {obt_table}
GROUP BY 1,2,3
ORDER BY 1,2,3
"""

df_cov = (
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_coverage)
    .load()
)

df_cov.show(60, truncate=False)

+------------+-----------+------------+--------+
|SERVICE_TYPE|SOURCE_YEAR|SOURCE_MONTH|ROWS_OBT|
+------------+-----------+------------+--------+
|green       |2015       |01          |1469403 |
|green       |2015       |02          |1536080 |
|green       |2015       |03          |1677899 |
|green       |2015       |04          |1620980 |
|green       |2015       |05          |1741602 |
|green       |2015       |06          |1597723 |
|green       |2015       |07          |1503389 |
|green       |2015       |08          |1495565 |
|green       |2015       |09          |1451661 |
|green       |2015       |10          |1581613 |
|green       |2015       |11          |1481843 |
|green       |2015       |12          |1558693 |
|green       |2016       |01          |1398910 |
|green       |2016       |02          |1465607 |
|green       |2016       |03          |1528739 |
|green       |2016       |04          |1497030 |
|green       |2016       |05          |1489462 |
|green       |2016  

In [20]:
expected_lotes = 11 * 12 * 2
real_lotes = df_cov.count()
print('expected_lotes =', expected_lotes)
print('real_lotes =', real_lotes)
print('coverage_ok =', real_lotes == expected_lotes)

expected_lotes = 264
real_lotes = 264
coverage_ok = True


## 6) Auditoria de ingesta por lote (RAW.INGESTION_AUDIT)

In [21]:
q_audit_status = f"""
SELECT status, COUNT(*) AS lotes
FROM {audit_table}
WHERE run_id = '{run_id}'
GROUP BY 1
ORDER BY 1
"""

(
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_audit_status)
    .load()
    .show(truncate=False)
)

+------+-----+
|STATUS|LOTES|
+------+-----+
|OK    |264  |
+------+-----+



In [22]:
q_audit_perf = f"""
SELECT
  service_type,
  ROUND(AVG(duration_sec), 2) AS avg_duration_sec,
  ROUND(MIN(duration_sec), 2) AS min_duration_sec,
  ROUND(MAX(duration_sec), 2) AS max_duration_sec,
  SUM(COALESCE(records_in, 0)) AS total_records_in,
  SUM(COALESCE(records_out, 0)) AS total_records_out
FROM {audit_table}
WHERE run_id = '{run_id}'
GROUP BY 1
ORDER BY 1
"""

(
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_audit_perf)
    .load()
    .show(truncate=False)
)

+------------+----------------+----------------+----------------+----------------+-----------------+
|SERVICE_TYPE|AVG_DURATION_SEC|MIN_DURATION_SEC|MAX_DURATION_SEC|TOTAL_RECORDS_IN|TOTAL_RECORDS_OUT|
+------------+----------------+----------------+----------------+----------------+-----------------+
|green       |14.08           |5.62            |36.87           |68239054        |68069321         |
|yellow      |50.67           |10.46           |325.94          |801553240       |248356367        |
+------------+----------------+----------------+----------------+----------------+-----------------+



## 7) Exploracion basica

In [23]:
q_top_month = f"""
SELECT year, month, service_type, COUNT(*) AS trips
FROM {obt_table}
GROUP BY 1,2,3
ORDER BY trips DESC
LIMIT 20
"""

(
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_top_month)
    .load()
    .show(truncate=False)
)

+----+-----+------------+--------+
|YEAR|MONTH|SERVICE_TYPE|TRIPS   |
+----+-----+------------+--------+
|2015|3    |yellow      |13193359|
|2015|5    |yellow      |13012441|
|2015|4    |yellow      |12922759|
|2015|1    |yellow      |12604548|
|2015|2    |yellow      |12308141|
|2015|6    |yellow      |12189546|
|2015|10   |yellow      |12166765|
|2016|3    |yellow      |12070630|
|2016|4    |yellow      |11789627|
|2016|5    |yellow      |11698020|
|2015|7    |yellow      |11432332|
|2015|12   |yellow      |11317899|
|2016|2    |yellow      |11252456|
|2015|11   |yellow      |11169245|
|2015|9    |yellow      |11086295|
|2016|6    |yellow      |11002669|
|2015|8    |yellow      |11002249|
|2016|1    |yellow      |10783249|
|2016|10   |yellow      |10727449|
|2016|12   |yellow      |10322300|
+----+-----+------------+--------+



In [24]:
q_top_borough = f"""
SELECT pu_borough, service_type, COUNT(*) AS trips
FROM {obt_table}
GROUP BY 1,2
ORDER BY trips DESC
LIMIT 20
"""

(
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_top_borough)
    .load()
    .show(truncate=False)
)

+-------------+------------+---------+
|PU_BOROUGH   |SERVICE_TYPE|TRIPS    |
+-------------+------------+---------+
|Manhattan    |yellow      |704276232|
|Queens       |yellow      |51726929 |
|Brooklyn     |green       |22734887 |
|Manhattan    |green       |21170371 |
|Queens       |green       |18421162 |
|Brooklyn     |yellow      |11802423 |
|Unknown      |yellow      |8764044  |
|Bronx        |green       |3611098  |
|Bronx        |yellow      |1333944  |
|N/A          |yellow      |257240   |
|Unknown      |green       |35183    |
|Staten Island|yellow      |28580    |
|N/A          |green       |19736    |
|Staten Island|green       |14288    |
|EWR          |yellow      |5548     |
|EWR          |green       |150      |
+-------------+------------+---------+

